---   
 <img align="left" width="75" height="75"  src="https://upload.wikimedia.org/wikipedia/en/c/c8/University_of_the_Punjab_logo.png"> 

<h1 align="center">Department of Data Science</h1>

---
<h3><div align="right">Instructor: Muhammad Arif Butt, Ph.D.</div></h3>    

<br><br>
<h1 align="center">Lec-36: LangChain and <b>Vector DBs</b></h1>

# Learning agenda of this notebook

> **The idea of this lecture:** after loading (Lec-34) and splitting (Lec-35), every chunk is turned into a vector and saved in a **vector store**, a database that can find the vectors closest to a question. In Lec-29 we used ChromaDB directly. LangChain puts **one interface** in front of many vector databases, so the same few lines work with Chroma, FAISS, PostgreSQL, Pinecone and many more.

1. **Where Vector Stores Fit**
2. **Setup: the Embedding Model and Some Documents**
3. **Storing and Searching**
    - Example 1: `InMemoryVectorStore`: the simplest store
    - Example 2: The same code with Chroma and FAISS, and what their scores mean
4. **Working with Stored Documents**
    - Example 3: Metadata filters
    - Example 4: Read, update and delete by id
    - Example 5: A persistent store: close it, open it again
5. **The Whole Indexing Pipeline**
    - Example 6: Load, split, embed and store a research paper, then ask questions
6. **Choosing a Vector Store**
7. **Summary**

> **Note:** everything runs locally with the free `all-MiniLM-L6-v2` embedding model: no API key is needed.

# <span style='background :lightgreen' >1. Where Vector Stores Fit</span>

Storing is the last step of preparing documents. After it, every question is answered by searching the store:

<div style="text-align:center;">
    <img src="../images/langchain-new-03.png"
         style="max-width:1500px; width:100%; height:auto; display:inline-block;">
</div>

**What we already know from Lec-29 (ChromaDB):**
- A vector database stores each text together with its **embedding** and its **metadata**.
- To search, the question is embedded with the **same model**, and the database returns the **nearest** vectors: by cosine, Euclidean (L2) or dot-product distance.
- To stay fast with millions of vectors, it uses an approximate index such as **HNSW**, built automatically when you add data.
- We called ChromaDB's own API: `client.get_or_create_collection()`, `collection.add()`, `collection.query()`, `collection.update()`, `collection.delete()`.

**What LangChain adds:** a common `VectorStore` interface. The methods are the same for every database, and the store calls the embedding model for us:

<div style="text-align:center;">
    <img src="../images/langchain-new-26.png"
         style="max-width:1500px; width:100%; height:auto; display:inline-block;">
</div>

# <span style='background :lightgreen' >2. Setup: the Embedding Model and Some Documents</span>

We use the free local model from Lec-30 (Example 14) and Lec-35: `all-MiniLM-L6-v2`, 384 numbers per text. The documents are short facts on three topics, each with metadata that we will filter on later.

In [1]:
import transformers                                                  # the library under sentence-transformers
from langchain_huggingface import HuggingFaceEmbeddings              # the free local embedding model
from langchain_core.documents import Document                        # the objects we store (Lec-34)

transformers.logging.set_verbosity_error()                           # hide library warnings
transformers.logging.disable_progress_bar()                          # hide the "Loading weights" progress bar

embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")   # runs on your computer

docs = [
    # geography
    Document("Islamabad is the capital of Pakistan", metadata={"category": "Geography", "country": "Pakistan"}),
    Document("Paris is the capital of France", metadata={"category": "Geography", "country": "France"}),
    Document("Rome is the capital of Italy", metadata={"category": "Geography", "country": "Italy"}),
    # LangChain and AI
    Document("LangChain makes it easy to work with LLMs.", metadata={"category": "AI Tools"}),
    Document("LangChain is used to build LLM based applications.", metadata={"category": "AI Tools"}),
    Document("Chroma is used to store and search document embeddings.", metadata={"category": "Vector Database"}),
    Document("Embeddings are vector representations of text.", metadata={"category": "AI Concepts"}),
    Document("MMR helps you get diverse results when doing similarity search.", metadata={"category": "Search"}),
    Document("LangChain supports Chroma, FAISS, Pinecone, and more.", metadata={"category": "AI Tools"}),
    # climate change
    Document("Climate change is causing glaciers to melt rapidly in the Arctic region.", metadata={"category": "Environment", "region": "Arctic"}),
    Document("Glaciers in the Arctic are melting at an alarming rate due to rising temperatures.", metadata={"category": "Environment", "region": "Arctic"}),
    Document("Deforestation in the Amazon is accelerating global climate change.", metadata={"category": "Environment", "region": "Amazon"}),
    Document("Climate change is increasing the frequency of wildfires in California.", metadata={"category": "Environment", "region": "California"}),
    Document("Rising sea levels due to climate change threaten coastal cities like Mumbai and New York.", metadata={"category": "Environment", "region": "Global"}),
]
ids = [f"doc-{i:02d}" for i in range(1, len(docs) + 1)]             # our own ids: doc-01 ... doc-14

QUESTION = "What is the capital of Pakistan?"
print(len(docs), "documents, ids", ids[0], "to", ids[-1])

14 documents, ids doc-01 to doc-14


# <span style='background :lightgreen' >3. Storing and Searching</span>

## <span style='color:#7e22ce'>Example 1: <code>InMemoryVectorStore</code>: the simplest store</span>

`InMemoryVectorStore` comes with `langchain-core`. It keeps the vectors in a Python list in RAM: no database, nothing to install, and everything is gone when Python stops. It is perfect for learning and for small tests.

In [2]:
from langchain_core.vectorstores import InMemoryVectorStore

memory_store = InMemoryVectorStore(embedding=embeddings)       # an empty store that knows which model to use
added = memory_store.add_documents(docs, ids=ids)              # embeds the 14 Documents and stores them
print("stored ids:", added[:3], "...")                         # add_documents() returns the ids

# 1. the closest Documents
for doc in memory_store.similarity_search(QUESTION, k=3):     # embeds the question, returns the 3 nearest
    print("  ", doc.page_content)

# 2. the same search, with a score for each Document
print()
for doc, score in memory_store.similarity_search_with_score(QUESTION, k=3):
    print(f"   {score:.4f}  {doc.page_content}")               # higher = more similar

stored ids: ['doc-01', 'doc-02', 'doc-03'] ...
   Islamabad is the capital of Pakistan
   Rome is the capital of Italy
   Paris is the capital of France

   0.8102  Islamabad is the capital of Pakistan
   0.3416  Rome is the capital of Italy
   0.2054  Paris is the capital of France


The scores are the same cosine similarities as in Lec-30 (figure 10): **0.8102** for Islamabad. Notice that the search returned `Document` objects, with their metadata: exactly what a RAG app needs.

## <span style='color:#7e22ce'>Example 2: The same code with Chroma and FAISS, and what their scores mean</span>

Switching to another database changes **only the line that creates the store**. We build three more stores with the same Documents:
- **Chroma** with the cosine distance (`"hnsw:space": "cosine"`, as in Lec-29);
- **Chroma** with its default distance, squared L2;
- **FAISS**, a fast vector-search library from Meta (`from_documents()` creates the store and adds the Documents in one call).

In [3]:
from langchain_chroma import Chroma                            # LangChain's wrapper around ChromaDB (Lec-29)
from langchain_community.vectorstores import FAISS             # LangChain's wrapper around FAISS

chroma_cosine = Chroma(collection_name="capitals_cosine",      # a collection, kept in RAM (no persist_directory)
                       embedding_function=embeddings,
                       collection_metadata={"hnsw:space": "cosine"})   # use the cosine distance
chroma_cosine.add_documents(docs, ids=ids)

chroma_l2 = Chroma(collection_name="capitals_l2", embedding_function=embeddings)   # Chroma's default: squared L2
chroma_l2.add_documents(docs, ids=ids)

faiss_store = FAISS.from_documents(docs, embeddings, ids=ids)  # FAISS: create and add in one call (default: L2)

def show(store, name):
    """Print the top 3 results and their scores for QUESTION."""
    print(name)
    for doc, score in store.similarity_search_with_score(QUESTION, k=3):   # the same call for every store
        print(f"   {float(score):.4f}  {doc.page_content}")

show(memory_store, "InMemoryVectorStore")
show(chroma_cosine, "Chroma, cosine")
show(chroma_l2, "Chroma, default (L2)")
show(faiss_store, "FAISS")

InMemoryVectorStore
   0.8102  Islamabad is the capital of Pakistan
   0.3416  Rome is the capital of Italy
   0.2054  Paris is the capital of France
Chroma, cosine
   0.1898  Islamabad is the capital of Pakistan
   0.6584  Rome is the capital of Italy
   0.7946  Paris is the capital of France
Chroma, default (L2)
   0.3796  Islamabad is the capital of Pakistan
   1.3168  Rome is the capital of Italy
   1.5892  Paris is the capital of France
FAISS
   0.3796  Islamabad is the capital of Pakistan
   1.3168  Rome is the capital of Italy
   1.5892  Paris is the capital of France


The **same order** everywhere, but four different numbers for Islamabad. `similarity_search_with_score()` returns whatever the database measures: a similarity (higher is better) or a distance (lower is better). For a number that always means "higher is better, between 0 and 1", use `similarity_search_with_relevance_scores()`:

In [4]:
for doc, score in chroma_cosine.similarity_search_with_relevance_scores(QUESTION, k=3):
    print(f"   {score:.4f}  {doc.page_content}")          # relevance = 1 - cosine distance

   0.8102  Islamabad is the capital of Pakistan
   0.3416  Rome is the capital of Italy
   0.2054  Paris is the capital of France


<div style="text-align:center;">
    <img src="../images/langchain-new-27.png"
         style="max-width:1500px; width:100%; height:auto; display:inline-block;">
</div>

# <span style='background :lightgreen' >4. Working with Stored Documents</span>

## <span style='color:#7e22ce'>Example 3: Metadata filters</span>

A search can be limited to Documents whose **metadata** matches a condition. The search then only compares the question with those Documents.

In [5]:
def show_docs(results):
    """Print each Document's text and its metadata."""
    for doc in results:
        print(f"   {doc.page_content}  {doc.metadata}")

question = "Which cities are in danger?"

print("1. no filter:")
show_docs(chroma_cosine.similarity_search(question, k=3))

print("\n2. only category = Environment:")
show_docs(chroma_cosine.similarity_search(question, k=3, filter={"category": "Environment"}))

print("\n3. only the Arctic or the Amazon:")
show_docs(chroma_cosine.similarity_search(question, k=3,
                                          filter={"region": {"$in": ["Arctic", "Amazon"]}}))   # $in: one of a list

1. no filter:
   Rising sea levels due to climate change threaten coastal cities like Mumbai and New York.  {'category': 'Environment', 'region': 'Global'}
   Rome is the capital of Italy  {'country': 'Italy', 'category': 'Geography'}
   Islamabad is the capital of Pakistan  {'category': 'Geography', 'country': 'Pakistan'}

2. only category = Environment:
   Rising sea levels due to climate change threaten coastal cities like Mumbai and New York.  {'region': 'Global', 'category': 'Environment'}
   Climate change is increasing the frequency of wildfires in California.  {'region': 'California', 'category': 'Environment'}
   Glaciers in the Arctic are melting at an alarming rate due to rising temperatures.  {'category': 'Environment', 'region': 'Arctic'}

3. only the Arctic or the Amazon:
   Glaciers in the Arctic are melting at an alarming rate due to rising temperatures.  {'region': 'Arctic', 'category': 'Environment'}
   Climate change is causing glaciers to melt rapidly in the Arctic 

Without a filter, the question about cities also returns two **capital cities**: "city" is close to "capital" in meaning, even though those facts say nothing about danger. With the filter, only the climate Documents are compared, and with `$in` only two regions. Chroma's filter language uses `{"key": value}` and operators such as `$in`, `$ne`, `$gt`, `$and` and `$or`.

> **Note:** the filter **syntax depends on the store**. Chroma and Pinecone use a dictionary like this one; `InMemoryVectorStore` takes a Python function, for example `filter=lambda doc: doc.metadata.get("region") == "Arctic"`.

## <span style='color:#7e22ce'>Example 4: Read, update and delete by id</span>

We gave every Document an id (`doc-01` ... `doc-14`). Ids let us work on single Documents, like the CRUD operations of Lec-29:

| Operation | LangChain | ChromaDB in Lec-29 |
|---|---|---|
| Create | `add_documents(docs, ids=...)` | `collection.add(...)` |
| Read | `get_by_ids([...])`, `similarity_search(...)` | `collection.get(...)`, `collection.query(...)` |
| Update | `update_document(id, doc)` | `collection.update(...)` |
| Delete | `delete(ids=[...])` | `collection.delete(...)` |

In [6]:
def count(store) -> int:
    """Number of Documents in a Chroma store."""
    return len(store.get()["ids"])                          # get() with no arguments returns all ids

print("read  :", chroma_cosine.get_by_ids(["doc-01"])[0].page_content)

new_text = Document("Islamabad has been the capital of Pakistan since 1967.",
                    metadata={"category": "Geography", "country": "Pakistan"})
chroma_cosine.update_document("doc-01", new_text)             # new text, so a new embedding too
print("update:", chroma_cosine.get_by_ids(["doc-01"])[0].page_content)

print("\ndelete: before", count(chroma_cosine), "Documents")
chroma_cosine.delete(ids=["doc-02"])                          # remove the Paris Document
print("        after ", count(chroma_cosine), "Documents")

chroma_cosine.add_documents(docs, ids=ids)                    # add all 14 again, with the same ids
print("\nadd the same ids again:", count(chroma_cosine), "Documents (no duplicates)")
print("doc-01 is back to:", chroma_cosine.get_by_ids(["doc-01"])[0].page_content)

read  : Islamabad is the capital of Pakistan
update: Islamabad has been the capital of Pakistan since 1967.

delete: before 14 Documents
        after  13 Documents

add the same ids again: 14 Documents (no duplicates)
doc-01 is back to: Islamabad is the capital of Pakistan


Adding a Document with an id that already exists **replaces** it (an *upsert*). That is why we always pass ids: you can run a cell again without filling the store with copies. Without ids, every run would add 14 new Documents with new random ids.

## <span style='color:#7e22ce'>Example 5: A persistent store: close it, open it again</span>

The stores so far live in RAM. With `persist_directory`, Chroma saves everything in a folder (Lec-29: `PersistentClient`). We create a store, delete the Python object, and open the folder again as if it were a new program.

In [7]:
import os                                                     # to look at the folder
import shutil                                                 # to delete the folder of an earlier run

DB_DIR = "../data/lec36_chroma_db"                            # this notebook creates this folder
shutil.rmtree(DB_DIR, ignore_errors=True)                     # start fresh on every run

# 1. create the store and add the Documents
db = Chroma(collection_name="facts", embedding_function=embeddings,
            persist_directory=DB_DIR,                         # save to this folder
            collection_metadata={"hnsw:space": "cosine"})
db.add_documents(docs, ids=ids)
print("saved:", count(db), "Documents")
del db                                                        # the Python object is gone; the folder stays

# 2. open the same folder again
db = Chroma(collection_name="facts", embedding_function=embeddings, persist_directory=DB_DIR)
print("opened again:", count(db), "Documents")
doc, score = db.similarity_search_with_score(QUESTION, k=1)[0]
print(f"best match: {doc.page_content} (cosine distance {score:.4f})")   # the cosine setting was saved too

print("\non disk:", sorted(os.listdir(DB_DIR)))               # a SQLite file and one folder per collection index
for name in os.listdir(DB_DIR):
    if os.path.isdir(os.path.join(DB_DIR, name)):            # the folder with the long name
        print(f"inside {name[:8]}...:", sorted(os.listdir(os.path.join(DB_DIR, name))))

saved: 14 Documents
opened again: 14 Documents
best match: Islamabad is the capital of Pakistan (cosine distance 0.1898)

on disk: ['7b34316b-7c3a-452a-be26-9a8b4af26ef9', 'chroma.sqlite3']
inside 7b34316b...: ['data_level0.bin', 'header.bin', 'length.bin', 'link_lists.bin']


The Documents, their vectors, their metadata and the collection's settings (cosine) all came back from the folder. As in Lec-29, `chroma.sqlite3` holds the documents and metadata, and the folder with the long name holds the HNSW index of the collection.

# <span style='background :lightgreen' >5. The Whole Indexing Pipeline</span>

## <span style='color:#7e22ce'>Example 6: Load, split, embed and store a research paper, then ask questions</span>

Now we put Lec-34, Lec-35 and this lecture together on `attention.pdf`:
1. **load** the 15 pages with `PyPDFLoader` (Lec-34, Example 4);
2. **split** them into chunks of at most 200 MiniLM tokens (Lec-35, Example 5);
3. **embed and store** the chunks in a persistent Chroma collection, next to the one from Example 5.

In [8]:
import time                                                          # to measure how long storing takes
from transformers import AutoTokenizer                               # to size chunks in MiniLM tokens
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

pages = PyPDFLoader("../data/attention.pdf").load()                  # 1. load: 15 Documents
splitter = RecursiveCharacterTextSplitter.from_huggingface_tokenizer(
    AutoTokenizer.from_pretrained("sentence-transformers/all-MiniLM-L6-v2"),
    chunk_size=200, chunk_overlap=20)
chunks = splitter.split_documents(pages)                             # 2. split: each chunk keeps its page

paper = Chroma(collection_name="attention_paper", embedding_function=embeddings,
               persist_directory=DB_DIR, collection_metadata={"hnsw:space": "cosine"})
chunk_ids = [f"attention-{i:03d}" for i in range(len(chunks))]      # attention-000, attention-001, ...
start = time.time()
paper.add_documents(chunks, ids=chunk_ids)                           # 3. embed and store
print(f"{len(pages)} pages -> {len(chunks)} chunks, embedded and stored in {time.time() - start:.1f} s")

15 pages -> 60 chunks, embedded and stored in 0.3 s


In [9]:
def ask(question, k=3, filter=None):
    """Print the k closest chunks: cosine distance, page number and the start of the text."""
    print("Q:", question)
    for doc, score in paper.similarity_search_with_score(question, k=k, filter=filter):
        text = doc.page_content[:90].replace("\n", " ")
        print(f"   {score:.3f}  page {doc.metadata['page_label']:>2} | {text}...")
    print()

ask("How many attention heads does the base model use?")
ask("What BLEU score did the big model get on English-to-German?")
ask("Who are the authors of the paper?", k=4)

Q: How many attention heads does the base model use?
   0.395  page  5 | dk = dv = dmodel/h = 64. Due to the reduced dimension of each head, the total computationa...
   0.464  page  5 | output values. These are concatenated and once again projected, resulting in the final val...
   0.496  page  3 | 3.2 Attention An attention function can be described as mapping a query and a set of key-v...

Q: What BLEU score did the big model get on English-to-German?
   0.393  page  8 | hurts perplexity, as the model learns to be more unsure, but improves accuracy and BLEU sc...
   0.437  page  8 | Table 2: The Transformer achieves better BLEU scores than previous state-of-the-art models...
   0.559  page  9 | Table 3: Variations on the Transformer architecture. Unlisted values are identical to thos...

Q: Who are the authors of the paper?
   0.785  page 11 | Information Processing Systems, (NIPS), 2016. [17] Łukasz Kaiser and Ilya Sutskever. Neura...
   0.787  page 12 | [25] Mitchell P Marcus, Mar

**Reading the results** (lower distance = closer):
- The first two questions find the right pages: the chunk about `h` heads on page 5, and the results on pages 8 and 9.
- The third one **fails**: the three closest chunks come from the reference list, because a list of names looks like "authors". The chunk with the real author names (page 1) is only **4th**: it also holds the copyright notice and the authors' e-mail addresses, which move its vector away from the question.

A vector store only finds **similar** text, not necessarily the **right** text. Lec-37 shows better ways to retrieve (more diverse results, several versions of the question, thresholds), and Lec-38 compares this with RAG that uses no vectors at all.

Metadata filters work here too, for example to search only page 8:

In [10]:
ask("BLEU score", k=2, filter={"page_label": "8"})       # page_label is a string ("8"), page counts from 0

Q: BLEU score
   0.522  page  8 | Table 2: The Transformer achieves better BLEU scores than previous state-of-the-art models...
   0.558  page  8 | hurts perplexity, as the model learns to be more unsure, but improves accuracy and BLEU sc...



# <span style='background :lightgreen' >6. Choosing a Vector Store</span>

| Store | Package | Runs | Keeps data | Good for |
|---|---|---|---|---|
| `InMemoryVectorStore` | `langchain-core` | in your Python process | no | learning, tests |
| `FAISS` | `langchain-community` + `faiss-cpu` | in your Python process | in a file, with `save_local()` | fast search, millions of vectors on one machine |
| `Chroma` | `langchain-chroma` | in your process, or as a server | yes (`persist_directory`) | local apps and prototypes |
| `PGVector` | `langchain-postgres` | inside PostgreSQL | yes | apps that already use PostgreSQL |
| `QdrantVectorStore`, `Pinecone`, MongoDB Atlas | `langchain-qdrant`, `langchain-pinecone`, `langchain-mongodb` | a server or a cloud service | yes | production, many users, very large data |

**Three rules for any store:**
1. **Embed and search with the same model.** Vectors from different models are not comparable, even when they have the same length. If you change the embedding model, embed all Documents again.
2. **Give every Document an id**, so that you can update and delete it, and re-run code without duplicates.
3. **Know what your score means** (similarity or distance) before you set thresholds (figure in Example 2).

# <span style='background :lightgreen' >7. Summary</span>

| Ex | What | Key methods |
|---|---|---|
| 1 | The simplest store, in RAM | `InMemoryVectorStore`, `add_documents`, `similarity_search(_with_score)` |
| 2 | Same code, other stores; similarity vs distance | `Chroma`, `FAISS.from_documents`, `similarity_search_with_relevance_scores` |
| 3 | Search only matching metadata | `filter={"region": {"$in": [...]}}` |
| 4 | CRUD by id | `get_by_ids`, `update_document`, `delete`, upsert |
| 5 | Save to disk and open again | `Chroma(persist_directory=...)` |
| 6 | Load, split, embed, store, ask | Lec-34 + Lec-35 + this lecture |

### Key takeaways
- A **vector store** saves each chunk with its embedding and metadata, and returns the chunks closest to a question.
- LangChain's **`VectorStore` interface** is the same for every database: changing the database changes one line.
- The store calls the embedding model itself: `embed_documents()` when adding, `embed_query()` when searching.
- `similarity_search_with_score()` returns a **similarity or a distance**, depending on the store; relevance scores are always "higher is better".
- Use **ids** for updates, deletes and safe re-runs, and **metadata filters** to search only part of the data.
- Similar is not always right: Example 6 shows a question that finds the wrong chunks.
- **Next lecture (Lec-37):** **Retrievers**: smarter ways to find the right chunks, starting with `as_retriever()`.

### Check your understanding
1. What does a vector store save for each chunk, and what does it return for a question?
2. Which line changes when you switch from `InMemoryVectorStore` to Chroma or FAISS?
3. For the same query, `InMemoryVectorStore` gives 0.8102 and Chroma (cosine) gives 0.1898. Why, and which one is better?
4. What does `similarity_search_with_relevance_scores()` return, and why is it useful?
5. How do you search only the Documents about the Arctic? Is the filter syntax the same in every store?
6. What happens when you add a Document with an id that is already in the store? Why does that make ids useful?
7. What do you need to change to make a Chroma store survive a restart of Python? Which files appear on disk?
8. In Example 6, why did "Who are the authors of the paper?" return the reference list?
9. Why must the same embedding model be used to add Documents and to search them?